In [1]:
import sys, re
import pandas as pd
import numpy as np

sys.path.append("../src")
import rag

# Build a flat table from the player documents' underlying data
games = pd.read_parquet("../data/raw/games.parquet")
box = pd.read_parquet("../data/raw/boxscores.parquet")

meta = games[["GAME_ID", "TEAM_ID", "GAME_DATE", "SEASON", "MATCHUP", "WL"]].drop_duplicates()
box = box.merge(meta, on=["GAME_ID", "TEAM_ID"], how="left")
box["IS_HOME"] = ~box["MATCHUP"].str.contains("@", na=False)
box["MIN_NUM"] = pd.to_numeric(box["MIN"].astype(str).str.split(":").str[0], errors="coerce")
box = box[box["SEASON"].notna() & box["MIN_NUM"].notna()]

print(f"{len(box):,} player-game rows")

W1103 02:26:55.698000 90289 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

25,775 player-game rows


In [2]:
def player_table(box):
    rows = []
    for (pid, season), g in box.groupby(["PLAYER_ID", "SEASON"]):
        if len(g) < 10:
            continue
        home, away = g[g["IS_HOME"]], g[~g["IS_HOME"]]
        rows.append({
            "player": g.iloc[0]["PLAYER_NAME"],
            "team": g.iloc[0]["TEAM_ABBREVIATION"],
            "season": season,
            "games": len(g),
            "ppg": g["PTS"].mean(),
            "rpg": g["REB"].mean(),
            "apg": g["AST"].mean(),
            "spg": g["STL"].mean(),
            "bpg": g["BLK"].mean(),
            "mpg": g["MIN_NUM"].mean(),
            "fg_pct": g["FGM"].sum() / g["FGA"].sum() * 100 if g["FGA"].sum() else np.nan,
            "fg3_pct": g["FG3M"].sum() / g["FG3A"].sum() * 100 if g["FG3A"].sum() else np.nan,
            "ft_pct": g["FTM"].sum() / g["FTA"].sum() * 100 if g["FTA"].sum() else np.nan,
            "plus_minus": g["PLUS_MINUS"].mean(),
            "home_ppg": home["PTS"].mean() if len(home) >= 3 else np.nan,
            "away_ppg": away["PTS"].mean() if len(away) >= 3 else np.nan,
            "max_pts": g["PTS"].max(),
            "max_reb": g["REB"].max(),
            "max_ast": g["AST"].max(),
        })
    df = pd.DataFrame(rows)
    df["home_away_gap"] = (df["home_ppg"] - df["away_ppg"]).abs()
    return df

PLAYERS = player_table(box)
PLAYERS.to_csv("../data/corpus/players_table.csv", index=False)
print(f"{len(PLAYERS)} player-seasons")
print(PLAYERS.head(3).to_string())

497 player-seasons
         player team   season  games        ppg       rpg       apg       spg       bpg        mpg     fg_pct    fg3_pct     ft_pct  plus_minus   home_ppg   away_ppg  max_pts  max_reb  max_ast  home_away_gap
0  LeBron James  LAL  2023-24     69  25.521739  7.318841  8.260870  1.217391  0.550725  34.898551  53.896104  40.281690  74.358974    2.507246  24.486486  26.718750     40.0     20.0     17.0       2.232264
1    Chris Paul  GSW  2023-24     57   9.315789  3.894737  6.789474  1.228070  0.105263  26.000000  44.276458  37.500000  82.692308    2.000000   9.928571   8.724138     24.0      9.0     14.0       1.204433
2    Kyle Lowry  MIA  2023-24     58   8.051724  3.241379  4.224138  0.982759  0.327586  27.568966  43.333333  39.662447  84.722222    0.051724   7.615385   8.406250     28.0     11.0     10.0       0.790865


In [3]:
games["IS_HOME"] = ~games["MATCHUP"].str.contains("@", na=False)

def team_table(games):
    rows = []
    for (tid, season), g in games.groupby(["TEAM_ID", "SEASON"]):
        wins = (g["WL"] == "W").sum()
        rows.append({
            "team": g.iloc[0]["TEAM_NAME"],
            "team_abbr": g.iloc[0]["TEAM_ABBREVIATION"],
            "season": season,
            "games": len(g),
            "wins": int(wins),
            "losses": int(len(g) - wins),
            "ppg": g["PTS"].mean(),
            "rpg": g["REB"].mean(),
            "apg": g["AST"].mean(),
            "point_diff": g["PLUS_MINUS"].mean(),
            "fg_pct": g["FGM"].sum() / g["FGA"].sum() * 100,
            "fg3_pct": g["FG3M"].sum() / g["FG3A"].sum() * 100,
            "max_pts": g["PTS"].max(),
        })
    return pd.DataFrame(rows)

TEAMS = team_table(games)
TEAMS.to_csv("../data/corpus/teams_table.csv", index=False)
print(f"{len(TEAMS)} team-seasons")
print(TEAMS.head(3).to_string())

60 team-seasons
             team team_abbr   season  games  wins  losses         ppg        rpg        apg  point_diff     fg_pct    fg3_pct  max_pts
0   Atlanta Hawks       ATL  2023-24     82    36      46  118.329268  44.670732  26.585366   -2.182927  46.532173  36.384217      152
1   Atlanta Hawks       ATL  2024-25     82    40      42  118.182927  44.500000  29.585366   -1.134146  47.236982  35.836840      148
2  Boston Celtics       BOS  2023-24     82    64      18  120.573171  46.329268  26.914634   11.341463  48.688480  38.799540      155


In [4]:
AGGREGATE_PATTERNS = [
    r"\bwhich (player|team|players|teams)\b",
    r"\bwho (had|led|was) the (most|best|highest|lowest|biggest)\b",
    r"\bbest\b.*\b(in the league|overall)\b",
    r"\b(most|best|highest|lowest|biggest|worst|top|leader|led the league)\b",
    r"\bhow many players\b",
    r"\bplayers (who|that) (averaged|scored|shot)\b",
]

def is_aggregate(q):
    ql = q.lower()
    return any(re.search(p, ql) for p in AGGREGATE_PATTERNS)

# Check it against your eval set
ev = pd.read_csv("../eval/eval_set.csv").fillna("")
ev["detected"] = ev["question"].apply(is_aggregate)
print(pd.crosstab(ev["behavior"], ev["detected"]).to_string())
print()
print("Detected as aggregate:")
print(ev[ev["detected"]][["qid", "behavior", "question"]].to_string(index=False))

detected   False  True 
behavior               
aggregate      0      4
answer        16      1
grounded       2      1
missing        8      0
refuse         7      1

Detected as aggregate:
 qid  behavior                                                                    question
  12 aggregate     Which Lakers player scored the most points in a single game in 2023-24?
  14    answer               What was the Indiana Pacers' highest-scoring game of 2023-24?
  21    refuse          Which player improved most in scoring between 2023-24 and 2024-25?
  25 aggregate  Which players averaged over 25 points while shooting above 50% in 2023-24?
  27 aggregate Which players had the biggest gap between home and road scoring in 2023-24?
  29 aggregate                      Which team had the best point differential in 2024-25?
  38  grounded                           Who was the best player in the league in 2023-24?


In [5]:
# Map words in the question to columns
PLAYER_METRICS = {
    "point": "ppg", "points": "ppg", "scoring": "ppg", "score": "ppg", "ppg": "ppg",
    "rebound": "rpg", "rebounds": "rpg", "rebounding": "rpg",
    "assist": "apg", "assists": "apg",
    "steal": "spg", "steals": "spg",
    "block": "bpg", "blocks": "bpg",
    "minutes": "mpg",
    "field goal": "fg_pct",
    "three": "fg3_pct", "3-point": "fg3_pct", "three-point": "fg3_pct",
    "free throw": "ft_pct",
    "plus-minus": "plus_minus", "plus minus": "plus_minus",
    "home and road": "home_away_gap", "home/road": "home_away_gap",
}
TEAM_METRICS = {
    "point differential": "point_diff", "differential": "point_diff",
    "win": "wins", "wins": "wins", "record": "wins",
    "point": "ppg", "points": "ppg", "scoring": "ppg",
    "rebound": "rpg", "rebounds": "rpg",
    "assist": "apg", "assists": "apg",
    "three": "fg3_pct", "three-point": "fg3_pct",
    "field goal": "fg_pct",
}

def pick_metric(q, mapping):
    ql = q.lower()
    hits = [(k, v) for k, v in mapping.items() if k in ql]
    if not hits:
        return None
    return max(hits, key=lambda kv: len(kv[0]))[1]   # longest match wins


def answer_aggregate(question, top_n=5):
    """Query the structured tables. Returns None when it shouldn't handle the question."""
    ql = question.lower()
    p = rag.parse_query(question)
    season = p["season"]

    # --- Guards: refuse anything the tables can't correctly answer ---

    # Single-game highs aren't in these tables (they hold per-season aggregates)
    if re.search(r"\bsingle game\b|\bin a game\b|\bhighest[- ]scoring game\b|\bbest game\b", ql):
        return None

    # Cross-season comparisons need two seasons of data per entity
    if re.search(r"\b(improved|improvement|between .* and |versus|vs\.?)\b", ql) and \
       len(re.findall(r"20\d\d[-–]\d\d", question)) > 1:
        return None

    # Questions scoped to one named team/player aren't league-wide rankings
    if p["players"] or p["teams"]:
        return None

    # Purely subjective — no single metric defines "best player"
    if re.search(r"\bbest player\b", ql) and not re.search(
            r"\b(point|rebound|assist|steal|block|scoring|shooting)\b", ql):
        return None

    # --- Now decide team vs player and find the metric ---
    is_team = bool(re.search(r"\bteam", ql)) and not re.search(r"\bplayer", ql)
    df = TEAMS.copy() if is_team else PLAYERS.copy()
    metrics = TEAM_METRICS if is_team else PLAYER_METRICS
    entity = "team" if is_team else "player"

    metric = pick_metric(question, metrics)
    if metric is None:
        return None

    if season:
        df = df[df["season"] == season]
        if df.empty:
            avail = sorted((TEAMS if is_team else PLAYERS)["season"].unique())
            return {"handled": True,
                    "text": f"The dataset has no {entity} data for {season}. "
                            f"Available: {', '.join(avail)}."}

    # --- Threshold queries ---
    conditions = []
    for m in re.finditer(r"(over|above|more than|at least)\s+(\d+(?:\.\d+)?)\s*(%|percent)?", ql):
        val, is_pct = float(m.group(2)), bool(m.group(3))
        before = ql[max(0, m.start() - 35):m.start()]
        col = "fg_pct" if (is_pct or "shooting" in before) else metric
        conditions.append((col, val))

    if conditions:
        sub = df.copy()
        for col, val in conditions:
            sub = sub[sub[col] > val]
        conds = " and ".join(f"{c} > {v}" for c, v in conditions)
        if sub.empty:
            return {"handled": True,
                    "text": f"No {entity}s in the dataset meet {conds} "
                            f"(all {len(df)} {entity}-seasons checked)."}
        sub = sub.sort_values(metric, ascending=False).head(top_n)
        lines = [f"{r[entity]} ({r['season']}): {r[metric]:.1f} {metric}, "
                 f"{r['fg_pct']:.1f}% FG, {int(r['games'])} games"
                 for _, r in sub.iterrows()]
        return {"handled": True,
                "text": f"{len(sub)} of {len(df)} {entity}-seasons meet {conds}:\n"
                        + "\n".join(lines)}

    # --- Ranking queries ---
        # --- Ranking queries ---
    MIN_GAMES = {"home_away_gap": 40, "fg3_pct": 30, "fg_pct": 30, "ft_pct": 30}
    floor = MIN_GAMES.get(metric)
    note = ""
    if floor and entity == "player":
        before = len(df)
        df = df[df["games"] >= floor]
        note = f" (minimum {floor} games; {before - len(df)} {entity}-seasons excluded)"

    ascending = bool(re.search(r"\b(lowest|worst|fewest|least)\b", ql))
    ranked = df.dropna(subset=[metric]).sort_values(metric, ascending=ascending).head(top_n)
    ascending = bool(re.search(r"\b(lowest|worst|fewest|least)\b", ql))
    ranked = df.dropna(subset=[metric]).sort_values(metric, ascending=ascending).head(top_n)
    lines = [f"{i+1}. {r[entity]} ({r['season']}): {r[metric]:.1f}"
             + (f" ({int(r['games'])} games)" if entity == "player" else "")
             for i, (_, r) in enumerate(ranked.iterrows())]
    return {"handled": True,
        "text": f"Ranked by {metric} ({'lowest' if ascending else 'highest'} first) "
                f"across all {len(df)} {entity}-seasons in the dataset{note}:\n"
                + "\n".join(lines)}

In [6]:
for qid in [12, 14, 21, 25, 27, 29, 38]:
    q = ev[ev["qid"] == qid]["question"].iloc[0]
    out = answer_aggregate(q)
    print(f"[{qid}] {q}")
    print(out["text"] if out else "  -> not handled, falls back to retrieval")
    print("-" * 70)

[12] Which Lakers player scored the most points in a single game in 2023-24?
  -> not handled, falls back to retrieval
----------------------------------------------------------------------
[14] What was the Indiana Pacers' highest-scoring game of 2023-24?
  -> not handled, falls back to retrieval
----------------------------------------------------------------------
[21] Which player improved most in scoring between 2023-24 and 2024-25?
  -> not handled, falls back to retrieval
----------------------------------------------------------------------
[25] Which players averaged over 25 points while shooting above 50% in 2023-24?
5 of 497 player-seasons meet ppg > 25.0 and fg_pct > 50.0:
Joel Embiid (2023-24): 34.2 ppg, 52.5% FG, 37 games
Giannis Antetokounmpo (2023-24): 30.3 ppg, 60.9% FG, 71 games
Shai Gilgeous-Alexander (2023-24): 29.9 ppg, 53.4% FG, 73 games
Kevin Durant (2023-24): 27.0 ppg, 52.1% FG, 74 games
Nikola Jokić (2023-24): 26.5 ppg, 59.4% FG, 77 games
----------------------

In [7]:
q = ev[ev["qid"] == 29]["question"].iloc[0]
print(q)
print(answer_aggregate(q)["text"])

Which team had the best point differential in 2024-25?
Ranked by point_diff (highest first) across all 30 team-seasons in the dataset:
1. Oklahoma City Thunder (2024-25): 12.9
2. Cleveland Cavaliers (2024-25): 9.5
3. Boston Celtics (2024-25): 9.1
4. Minnesota Timberwolves (2024-25): 5.0
5. Memphis Grizzlies (2024-25): 4.9


In [8]:
def answer_routed(question, k=3):
    """Try the structured tables first; fall back to retrieval."""
    if is_aggregate(question):
        agg = answer_aggregate(question)
        if agg is not None:
            return {
                "question": question,
                "answer": agg["text"],
                "route": "structured",
                "sources": ["players_table" if "player" in agg["text"] else "teams_table"],
                "source_names": ["structured query"],
                "context": agg["text"],
                "notes": [],
            }
    out = rag.answer(question, k)
    out["route"] = "retrieval"
    return out


# Quick check on the three fixed questions
for qid in [25, 27, 29]:
    q = ev[ev["qid"] == qid]["question"].iloc[0]
    r = answer_routed(q)
    print(f"[{qid}] route={r['route']}")
    print(r["answer"][:200])
    print("-" * 70)

[25] route=structured
5 of 497 player-seasons meet ppg > 25.0 and fg_pct > 50.0:
Joel Embiid (2023-24): 34.2 ppg, 52.5% FG, 37 games
Giannis Antetokounmpo (2023-24): 30.3 ppg, 60.9% FG, 71 games
Shai Gilgeous-Alexander (20
----------------------------------------------------------------------
[27] route=structured
Ranked by home_away_gap (highest first) across all 347 player-seasons in the dataset (minimum 40 games; 150 player-seasons excluded):
1. Donovan Mitchell (2023-24): 5.7 (53 games)
2. Jordan Clarkson (
----------------------------------------------------------------------
[29] route=structured
Ranked by point_diff (highest first) across all 30 team-seasons in the dataset:
1. Oklahoma City Thunder (2024-25): 12.9
2. Cleveland Cavaliers (2024-25): 9.5
3. Boston Celtics (2024-25): 9.1
4. Minne
----------------------------------------------------------------------


In [9]:
import time, re as _re

DECLINE = ["no information", "not provided", "not present", "not mention", "does not mention",
           "doesn't mention", "not contain", "doesn't contain", "does not contain",
           "does not include", "doesn't include", "no data", "not available",
           "cannot", "can't", "unable", "only covers", "only contains", "no documents",
           "no player", "no team"]

def numbers_in(text):
    text = _re.sub(r"[Dd]ocument \d+", " ", text)
    text = _re.sub(r"20\d\d[-–]\d\d", " ", text)
    text = _re.sub(r"\b(19|20)\d\d\b", " ", text)
    return set(_re.findall(r"\d+(?:\.\d+)?", text))

rows, t0 = [], time.time()
for _, q in ev.iterrows():
    r = answer_routed(q["question"])
    rows.append({
        "qid": q["qid"], "tier": q["tier"], "behavior": q["behavior"],
        "question": q["question"], "answer": r["answer"],
        "route": r["route"], "retrieved": "; ".join(r["source_names"]),
        "declined": any(k in r["answer"].lower() for k in DECLINE),
    })
    print(f"[{q['qid']:>2}] {r['route']:>10}  {time.time()-t0:5.0f}s")

res2 = pd.DataFrame(rows)
res2.to_csv("../eval/results_routed.csv", index=False)
print(f"\nDone in {time.time()-t0:.0f}s")
print(res2["route"].value_counts().to_string())

[ 1]  retrieval      1s
[ 2]  retrieval      1s
[ 3]  retrieval      2s
[ 4]  retrieval      2s
[ 5]  retrieval      2s
[ 6]  retrieval      3s
[ 7]  retrieval      3s
[ 8]  retrieval      4s
[ 9]  retrieval      4s
[10]  retrieval      5s
[11]  retrieval      7s
[12]  retrieval      7s
[13]  retrieval      8s
[14]  retrieval      9s
[15]  retrieval     11s
[16]  retrieval     12s
[17]  retrieval     12s
[18]  retrieval     13s
[19]  retrieval     14s
[20]  retrieval     15s
[21]  retrieval     16s
[22]  retrieval     17s
[23]  retrieval     18s
[24]  retrieval     19s
[25] structured     19s
[26]  retrieval     21s
[27] structured     21s
[28]  retrieval     22s
[29] structured     22s
[30]  retrieval     23s
[31]  retrieval     24s
[32]  retrieval     24s
[33]  retrieval     24s
[34]  retrieval     25s
[35]  retrieval     25s
[36]  retrieval     26s
[37]  retrieval     27s
[38]  retrieval     30s
[39]  retrieval     31s
[40]  retrieval     32s

Done in 32s
route
retrieval     37
stru

In [10]:
print(open("../src/llm.py").read())

"""One interface for text generation. Swap backends via .env."""
import os
from dotenv import load_dotenv

load_dotenv()
BACKEND = os.getenv("LLM_BACKEND", "ollama")


def generate(prompt: str, max_tokens: int = 500) -> str:
    """Send a prompt to the configured LLM and return the text response."""
    if BACKEND == "ollama":
        import ollama
        r = ollama.chat(
            model=os.getenv("OLLAMA_MODEL", "llama3.1:8b"),
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0, "seed": 42},
        )
        return r["message"]["content"]

    from anthropic import Anthropic
    client = Anthropic()
    msg = client.messages.create(
        model=os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5"),
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text


In [11]:
import importlib, sys
sys.path.append("../src")
import llm
importlib.reload(llm)

p = "Answer in one sentence: what is 2+2?"
print(repr(llm.generate(p)))
print(repr(llm.generate(p)))
print(repr(llm.generate(p)))

'The answer is 4.'
'The answer is 4.'
'The answer is 4.'


In [12]:
import time, re as _re

DECLINE = ["no information", "not provided", "not present", "not mention", "does not mention",
           "doesn't mention", "not contain", "doesn't contain", "does not contain",
           "does not include", "doesn't include", "no data", "not available",
           "cannot", "can't", "unable", "only covers", "only contains", "no documents",
           "no player", "no team", "not break down", "doesn't break down"]

rows, t0 = [], time.time()
for _, q in ev.iterrows():
    r = answer_routed(q["question"])
    rows.append({
        "qid": q["qid"], "tier": q["tier"], "behavior": q["behavior"],
        "question": q["question"], "answer": r["answer"],
        "route": r["route"], "retrieved": "; ".join(r["source_names"]),
        "declined": any(k in r["answer"].lower() for k in DECLINE),
    })
    print(f"[{q['qid']:>2}] {r['route']:>10}  {time.time()-t0:5.0f}s")

res2 = pd.DataFrame(rows)
res2.to_csv("../eval/results_routed.csv", index=False)
print(f"\nDone in {time.time()-t0:.0f}s")
print(res2["route"].value_counts().to_string())

[ 1]  retrieval      1s
[ 2]  retrieval      1s
[ 3]  retrieval      2s
[ 4]  retrieval      2s
[ 5]  retrieval      3s
[ 6]  retrieval      3s
[ 7]  retrieval      4s
[ 8]  retrieval      4s
[ 9]  retrieval      4s
[10]  retrieval      5s
[11]  retrieval      6s
[12]  retrieval      7s
[13]  retrieval      8s
[14]  retrieval      8s
[15]  retrieval     11s
[16]  retrieval     11s
[17]  retrieval     12s
[18]  retrieval     13s
[19]  retrieval     14s
[20]  retrieval     14s
[21]  retrieval     16s
[22]  retrieval     17s
[23]  retrieval     18s
[24]  retrieval     19s
[25] structured     19s
[26]  retrieval     20s
[27] structured     20s
[28]  retrieval     22s
[29] structured     22s
[30]  retrieval     23s
[31]  retrieval     23s
[32]  retrieval     23s
[33]  retrieval     24s
[34]  retrieval     24s
[35]  retrieval     25s
[36]  retrieval     25s
[37]  retrieval     26s
[38]  retrieval     30s
[39]  retrieval     31s
[40]  retrieval     32s

Done in 32s
route
retrieval     37
stru

In [13]:
GRADES = {
    1: "correct", 2: "correct", 3: "correct", 4: "correct", 5: "correct",
    6: "correct", 7: "correct", 8: "correct", 9: "correct", 10: "correct",
    11: "substituted", 12: "correct_refusal", 13: "correct_refusal", 14: "correct",
    15: "correct_refusal", 16: "correct_refusal", 17: "correct_refusal", 18: "correct",
    19: "correct", 20: "correct", 21: "correct_refusal", 22: "correct",
    23: "wrong", 24: "correct_refusal", 25: "sample_as_population", 26: "fabricated",
    27: "sample_as_population", 28: "substituted", 29: "sample_as_population",
    30: "correct", 31: "correct_refusal", 32: "correct_refusal", 33: "correct_refusal",
    34: "correct_refusal", 35: "correct_refusal", 36: "correct_refusal",
    37: "correct_refusal", 38: "correct", 39: "correct", 40: "correct",
}

g = pd.read_csv("../eval/grading_sheet.csv")
g["grade"] = g["qid"].map(GRADES)
g.to_csv("../eval/grading_sheet.csv", index=False)
print(g["grade"].value_counts().to_string())
print(f"\nCorrect: {g['grade'].isin(['correct','correct_refusal']).sum()}/40")

grade
correct                 19
correct_refusal         14
sample_as_population     3
substituted              2
wrong                    1
fabricated               1

Correct: 33/40


In [14]:
import time
rows, t0 = [], time.time()
for _, q in ev.iterrows():
    r = rag.answer(q["question"])          # retrieval only, no router
    rows.append({
        "qid": q["qid"], "tier": q["tier"], "behavior": q["behavior"],
        "question": q["question"], "answer": r["answer"],
        "route": "retrieval", "retrieved": "; ".join(r["source_names"]),
    })
    print(f"[{q['qid']:>2}] {time.time()-t0:5.0f}s")

pd.DataFrame(rows).to_csv("../eval/results_baseline_det.csv", index=False)
print("done")


[ 1]     1s
[ 2]     1s
[ 3]     2s
[ 4]     2s
[ 5]     3s
[ 6]     3s
[ 7]     4s
[ 8]     4s
[ 9]     4s
[10]     5s
[11]     6s
[12]     7s
[13]     8s
[14]     8s
[15]    11s
[16]    12s
[17]    12s
[18]    13s
[19]    14s
[20]    15s
[21]    16s
[22]    17s
[23]    18s
[24]    18s
[25]    20s
[26]    21s
[27]    24s
[28]    26s
[29]    27s
[30]    28s
[31]    28s
[32]    28s
[33]    29s
[34]    29s
[35]    30s
[36]    30s
[37]    31s
[38]    35s
[39]    36s
[40]    37s
done


In [15]:
b = pd.read_csv("../eval/results_baseline_det.csv")
r = pd.read_csv("../eval/results_routed.csv")
m = b[["qid","answer"]].rename(columns={"answer":"base"}).merge(r[["qid","answer","route"]], on="qid")
m["changed"] = m["base"].str.strip() != m["answer"].str.strip()
print(m.groupby("route")["changed"].agg(["sum","size"]).to_string())

            sum  size
route                
retrieval     0    37
structured    3     3


In [16]:
GRADES_ROUTED = dict(GRADES)
GRADES_ROUTED.update({25: "correct", 27: "correct", 29: "correct"})

r = pd.read_csv("../eval/results_routed.csv")
r["grade"] = r["qid"].map(GRADES_ROUTED)
r.to_csv("../eval/grading_sheet_routed.csv", index=False)

good = r["grade"].isin(["correct", "correct_refusal"])
print(f"After router: {good.sum()}/40 ({good.mean():.0%})")
print()
comp = pd.DataFrame({
    "before": g["grade"].isin(["correct","correct_refusal"]).groupby(g["behavior"]).sum(),
    "after": good.groupby(r["behavior"]).sum(),
    "n": g.groupby("behavior").size(),
})
print(comp.to_string())
comp.to_csv("../eval/before_after.csv")

After router: 36/40 (90%)

           before  after   n
behavior                    
aggregate       1      4   4
answer         16     16  17
grounded        3      3   3
missing         5      5   8
refuse          8      8   8
